# Correcting a segmentation in the Cellpose GUI (one annotation round)

Runs on the **Mac** (the GUI needs a display). Copies a `.zarr` written by `macroseg.baseline` on Mewtwo,
exports one segmentation for the Cellpose GUI and opens it. Each image of each round gets its own folder,
`$MACROSEG_ANNOTATIONS/round_<NN>/<name>/` (see `macroseg.annotation`):

- `<name>.tif`: three image channels, shown as red, green and blue (the GUI keeps only the first three);
- `<name>_seg.npy`: the masks, loaded by the GUI; Cmd + S saves the corrections into it;
- `start_masks.tif`: the masks as exported, to count the changes;
- `annotation.json`: round, role (`train` / `test`), where image and masks come from, and after section 4
  what was changed and how well the starting masks scored.

Paths come from `.env` (see `.env.example`): `MACROSEG_RUNS` and `MACROSEG_ANNOTATIONS` are the local copies,
`MACROSEG_REMOTE_RUNS` and `MACROSEG_REMOTE_ANNOTATIONS` the folders on Mewtwo.

In the GUI: right-click and draw to add a cell, Cmd + click to delete one, Cmd + S to save.
Review the **whole** image: untouched cells count as correct.


In [ ]:
import subprocess
import sys
from pathlib import Path

from spatialdata import read_zarr

from macroseg import paths
from macroseg.annotation import export_for_gui, finish
from macroseg.config import load_config

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "exp201_test_crop.yaml")

ROUND = 1                                   # annotation round
ROLE = "train"                              # "train" or "test" (a test image is never used for training)
SEGMENTATION = "membrane_dapi"              # starting masks: a labels element of the store
CHANNELS = ["F480", "NaK_ATPase", "DAPI"]   # red, green, blue in the GUI
OVERWRITE = False                           # True: export again and discard the corrections saved in the GUI

ANNOTATIONS = paths.path("MACROSEG_ANNOTATIONS")

## 1. Copy the store from Mewtwo
From `MACROSEG_REMOTE_RUNS` on Mewtwo to `cfg.zarr`, the local copy (in `MACROSEG_RUNS`).


In [ ]:
local_zarr = Path(cfg.zarr)
local_zarr.parent.mkdir(parents=True, exist_ok=True)
remote_zarr = f"{paths.value('MACROSEG_REMOTE_RUNS')}/{local_zarr.name}"
subprocess.run(["rsync", "-a", remote_zarr, f"{local_zarr.parent}/"], check=True)
sdata = read_zarr(local_zarr)
sdata

## 2. Export image and masks
If the folder already holds `<name>_seg.npy`, nothing is exported (it may hold your corrections) unless `OVERWRITE = True`.


In [ ]:
folder = export_for_gui(sdata, cfg, SEGMENTATION, CHANNELS, ANNOTATIONS, ROUND, role=ROLE, overwrite=OVERWRITE)
image_file = folder / f"{cfg.name}.tif"
folder

## 3. Open the Cellpose GUI
Started as a separate process, so the notebook stays free and closing the GUI does not stop the kernel.


In [ ]:
subprocess.Popen([sys.executable, "-c", f"from cellpose.gui import gui; gui.run(image={str(image_file)!r})"])

## 4. After annotating: record the changes and copy to Mewtwo
Save in the GUI first (Cmd + S). `complete=False` if the image was not reviewed entirely.
Can be run again after further corrections.


In [ ]:
record = finish(folder, complete=True)
record["counts"], record["start_score"]

In [ ]:
host, remote = paths.value("MACROSEG_REMOTE_ANNOTATIONS").split(":", 1)
remote = f"{remote}/round_{ROUND:02d}/"
subprocess.run(["ssh", host, f"mkdir -p {remote}"], check=True)
subprocess.run(["rsync", "-a", str(folder), f"{host}:{remote}"], check=True)
print(f"copied to {host}:{remote}{cfg.name}")